## Final Project Submission

Please fill out:
* Student name: 
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: 
* Blog post URL:


In [7]:
from pathlib import Path
import zipfile, sqlite3
import pandas as pd

DATA_DIR = Path("zippedData")
IMDB_ZIP = DATA_DIR / "im.db.zip"
IMDB_DB  = DATA_DIR / "im.db"

In [8]:
# Box Office Mojo
bom = pd.read_csv(DATA_DIR / "bom.movie_gross.csv.gz", low_memory=False)
bom.head()      # preview
bom.shape       # rows, columns
bom.columns     # column names

Index(['title', 'studio', 'domestic_gross', 'foreign_gross', 'year'], dtype='object')

In [9]:
# The Numbers budgets
tn_budgets = pd.read_csv(DATA_DIR / "tn.movie_budgets.csv.gz", low_memory=False)
tn_budgets.head()

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"


In [11]:
# Rotten Tomatoes (TSV)
rt_info = pd.read_csv(DATA_DIR / "rt.movie_info.tsv.gz", sep="\t", low_memory=False)
rt_info.head()

,id,synopsis,rating,genre,director,writer,theater_date,dvd_date,currency,box_office,runtime,studio
0,1,"This gritty, fast-paced, and innovative police...",R,Action and Adventure|Classics|Drama,William Friedkin,Ernest Tidyman,"Oct 9, 1971","Sep 25, 2001",NaN,NaN,104 minutes,NaN
1,3,"New York City, not-too-distant-future: Eric Pa...",R,Drama|Science Fiction and Fantasy,David Cronenberg,David Cronenberg|Don DeLillo,"Aug 17, 2012","Jan 1, 2013",$,"600,000",108 minutes,Entertainment One
2,5,Illeana Douglas delivers a superb performance ...,R,Drama|Musical and Performing Arts,Allison Anders,Allison Anders,"Sep 13, 1996","Apr 18, 2000",NaN,NaN,116 minutes,NaN
3,6,Michael Douglas runs afoul of a treacherous su...,R,Drama|Mystery and Suspense,Barry Levinson,Paul Attanasio|Michael Crichton,"Dec 9, 1994","Aug 27, 1997",NaN,NaN,128 minutes,NaN
4,7,NaN,NR,Drama|Romance,Rodney Bennett,Giles Cooper,NaN,NaN,NaN,NaN,200 minutes,NaN


In [12]:
tmdb = pd.read_csv(DATA_DIR / "tmdb.movies.csv.gz", low_memory=False)
tmdb.head()

,Unnamed: 0,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count
0,0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788
1,1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610
2,2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368
3,3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174
4,4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186


In [13]:
if not IMDB_DB.exists():
    with zipfile.ZipFile(IMDB_ZIP, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Unzipped:", IMDB_DB)

In [14]:
conn = sqlite3.connect(str(IMDB_DB))

# Basics table (name varies across datasets)
try:
    imdb_basics = pd.read_sql("SELECT * FROM movie_basics", conn)
except Exception:
    imdb_basics = pd.read_sql("SELECT * FROM title_basics", conn)

imdb_basics.head()
# Ratings table (name varies)
try:
    imdb_ratings = pd.read_sql("SELECT * FROM movie_ratings", conn)
except Exception:
    imdb_ratings = pd.read_sql("SELECT * FROM title_ratings", conn)

imdb_ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [16]:
with sqlite3.connect("zippedData/im.db") as conn:
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';",
        conn
    )
    print("Table count:", len(tables))
    display(tables)

Table count: 8


,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [18]:
db_path = "zippedData/im.db"

with sqlite3.connect(db_path) as conn:
    # Tables (excluding SQLite internals)
    tables = pd.read_sql("""
        SELECT name FROM sqlite_master 
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name;
    """, conn)
    display(tables)

    # Row counts
    print("\nRow counts:")
    for t in tables["name"]:
        n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t};", conn)["n"][0]
        print(f"{t:15s} {n:,}")

    # Columns per table
    print("\nSchemas:")
    for t in tables["name"]:
        print(f"\n-- {t} --")
        display(pd.read_sql(f"PRAGMA table_info({t});", conn))


,name
0,directors
1,known_for
2,movie_akas
3,movie_basics
4,movie_ratings
5,persons
6,principals
7,writers



Row counts:
directors       291,174
known_for       1,638,260
movie_akas      331,703
movie_basics    146,144
movie_ratings   73,856
persons         606,648
principals      1,028,186
writers         255,873

Schemas:

-- directors --


,cid,name,type,notnull,dflt_value,pk
0,0,movie_id,TEXT,0,None,0
1,1,person_id,TEXT,0,None,0



-- known_for --


,cid,name,type,notnull,dflt_value,pk
0,0,person_id,TEXT,0,None,0
1,1,movie_id,TEXT,0,None,0


...

,cid,name,type,notnull,dflt_value,pk
0,0,person_id,TEXT,0,None,0
1,1,primary_name,TEXT,0,None,0
2,2,birth_year,REAL,0,None,0
3,3,death_year,REAL,0,None,0
4,4,primary_profession,TEXT,0,None,0



-- principals --


,cid,name,type,notnull,dflt_value,pk
0,0,movie_id,TEXT,0,None,0
1,1,ordering,INTEGER,0,None,0
2,2,person_id,TEXT,0,None,0
3,3,category,TEXT,0,None,0
4,4,job,TEXT,0,None,0
5,5,characters,TEXT,0,None,0



-- writers --


,cid,name,type,notnull,dflt_value,pk
0,0,movie_id,TEXT,0,None,0
1,1,person_id,TEXT,0,None,0


In [19]:
with sqlite3.connect(db_path) as conn:
    df_titles = pd.read_sql("""
        SELECT b.tconst,
               b.primary_title,
               b.start_year,
               b.runtime_minutes,
               b.genres,
               r.average_rating,
               r.num_votes
        FROM movie_basics b
        LEFT JOIN movie_ratings r USING (tconst)
    """, conn)

df_titles.head()

DatabaseError: Execution failed on sql '
        SELECT b.tconst,
               b.primary_title,
               b.start_year,
               b.runtime_minutes,
               b.genres,
               r.average_rating,
               r.num_votes
        FROM movie_basics b
        LEFT JOIN movie_ratings r USING (tconst)
    ': cannot join using column tconst - column not present in both tables